# Atividade Prática — Aula 3: Limpeza, Preparação e Qualidade dos Dados com Pandas

Esta atividade foi construída com base nos slides da Aula 3, que destacam a **limpeza como a fundação invisível** de qualquer dashboard confiável. O objetivo aqui não é apenas "fazer funcionar", mas tomar decisões conscientes sobre tipos, valores ausentes, duplicidades, variáveis derivadas e exportação da base limpa. fileciteturn2file0

## Regras desta atividade
- Você deve **construir os códigos**.
- O notebook orienta os passos, mas não entrega as soluções prontas.
- Sempre que fizer uma decisão de limpeza, **documente o porquê** em comentário ou célula markdown.
- Ao final, exporte uma base limpa para uso nas próximas aulas.

## Dataset desta atividade
Arquivo bruto: `vendas_brasil_aula3_bruto.csv`


## 1. Preparação do ambiente

Importe as bibliotecas necessárias para trabalhar com:
- leitura de dados
- tratamento de nulos
- identificação de infinitos
- exportação do resultado final

**Sugestão:** Pandas e NumPy já resolvem toda a atividade.


In [2]:
import pandas as pd
import numpy as np

## 2. Leitura da base bruta

Leia o arquivo `vendas_brasil_aula3_bruto.csv` em um DataFrame chamado `df`.

Depois:
1. Exiba as primeiras linhas
2. Exiba as últimas linhas
3. Observe visualmente se já existem sinais de sujeira


In [3]:
df = pd.read_csv('vendas_brasil_aula3_bruto.csv')

# Exiba as primeiras linhas
print("Primeiras 5 linhas do DataFrame:")
display(df.head())

# Exiba as últimas linhas
print("\nÚltimas 5 linhas do DataFrame:")
display(df.tail())

Primeiras 5 linhas do DataFrame:


,pedido_id,data,uf,canal,categoria,produto,cliente_id,quantidade,receita,lucro,observacao
0,1000,2024/04/29,SP,Loja Física,Informática,Notebook Pro,C134,1,4535.11,1289.83,ok
1,1001,2024-06-17,PR,NaN,Informática,Notebook Pro,C106,1,4005.59,541.36,entrega rápida
2,1002,2024-05-27,PR,Marketplace,Periféricos,Teclado Mecânico,C131,1,309.02,128.26,entrega rápida
3,1003,2024-07-08,SP,Marketplace,Informática,Notebook Pro,C148,2,7943.78,1574.09,ok
4,1004,2024-05-20,RS,Online,Informática,Notebook Pro,C105,5,19926.65,4281.84,ok



Últimas 5 linhas do DataFrame:


,pedido_id,data,uf,canal,categoria,produto,cliente_id,quantidade,receita,lucro,observacao
223,1180,2024-03-11,SC,Loja Física,Telefonia,Smartphone X,C158,1,2563.44,415.42,NaN
224,1015,2024-02-26,RS,Online,Informática,Notebook Pro,C102,3,15011.49,NaN,cliente premium
225,1115,2024-07-29,MG,Loja Física,Informática,Notebook Pro,C144,3,12566.04,2879.76,NaN
226,1172,2024-11-11,ES,Loja Física,Informática,Notebook Pro,C128,3,13154.04,3408.22,entrega rápida
227,1209,2024-05-27,MG,Marketplace,Áudio,Caixa de Som,C121,2,810.2,282.52,NaN


## 3. Check-up inicial do dataset

Com base no checklist de um dataset "clean" apresentado na aula, faça um diagnóstico inicial da base. fileciteturn2file0

### Sua missão
Use comandos que permitam responder:
- Qual é o tamanho da base?
- Quais são os tipos atuais das colunas?
- Existem valores nulos?
- Há colunas com tipo inadequado?
- Há algo que pareça impedir análises confiáveis?


In [4]:
# Faça aqui o check-up inicial: shape, info, dtypes, isna, etc.

print("Tamanho da base (linhas, colunas):")
print(df.shape)

print("\nInformações gerais do DataFrame (tipos e nulos):")
df.info()

print("\nContagem de valores nulos por coluna:")
print(df.isnull().sum())

Tamanho da base (linhas, colunas):
(228, 11)

Informações gerais do DataFrame (tipos e nulos):
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 228 entries, 0 to 227
Data columns (total 11 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   pedido_id   228 non-null    int64  
 1   data        228 non-null    object 
 2   uf          223 non-null    object 
 3   canal       223 non-null    object 
 4   categoria   223 non-null    object 
 5   produto     228 non-null    object 
 6   cliente_id  228 non-null    object 
 7   quantidade  228 non-null    int64  
 8   receita     226 non-null    object 
 9   lucro       222 non-null    float64
 10  observacao  178 non-null    object 
dtypes: float64(1), int64(2), object(8)
memory usage: 19.7+ KB

Contagem de valores nulos por coluna:
pedido_id      0
data           0
uf             5
canal          5
categoria      5
produto        0
cliente_id     0
quantidade     0
receita        2
lucro       

## 4. Batalha 1 — A tirania dos tipos de dados

Nos slides, vimos que datas não podem ser tratadas como texto e que números em formato string quebram análises. fileciteturn2file0

### Tarefa
Converta corretamente, quando fizer sentido:
- `data`
- `receita`
- `lucro`
- `quantidade`

### Orientação
- Investigue valores estranhos antes de converter
- Pense no uso de `errors='coerce'`
- Registre em markdown ou comentário quais problemas encontrou


In [5]:

df['data'] = pd.to_datetime(df['data'], infer_datetime_format=True, errors='coerce')
print("Valores únicos na coluna 'data' após coerção (se houver NaT):\n", df['data'].unique()[:5])

receita_nao_numerica = df[pd.to_numeric(df['receita'], errors='coerce').isna() & df['receita'].notna()]['receita']

if not receita_nao_numerica.empty:
    print("\nValores não numéricos encontrados na coluna 'receita':")
    print(receita_nao_numerica.unique())
    df['receita'] = df['receita'].astype(str).str.replace(',', '', regex=False).str.replace('R$', '', regex=False).str.strip()

df['receita'] = pd.to_numeric(df['receita'], errors='coerce')

print("\nVerificando valores infinitos na coluna 'lucro':")
print(np.isinf(df['lucro']).sum())

print("\nVerificando valores nulos na coluna 'quantidade':")
print(df['quantidade'].isnull().sum())

print("\nInformações gerais do DataFrame após conversões de tipo:")
df.info()

Valores únicos na coluna 'data' após coerção (se houver NaT):
 <DatetimeArray>
['2024-04-29 00:00:00', 'NaT', '2024-09-30 00:00:00', '2024-10-21 00:00:00']
Length: 4, dtype: datetime64[ns]

Valores não numéricos encontrados na coluna 'receita':
['zero' 'R$ 748,80' 'R$ 4.507,32' 'R$ 672,24' 'R$ 2.685,80' 'R$ 7.947,06'
 'R$ 1.434,75' 'R$ 1.031,55' 'R$ 316,70' 'R$ 3.281,90' 'R$ 411,96']

Verificando valores infinitos na coluna 'lucro':
0

Verificando valores nulos na coluna 'quantidade':
0

Informações gerais do DataFrame após conversões de tipo:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 228 entries, 0 to 227
Data columns (total 11 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   pedido_id   228 non-null    int64         
 1   data        4 non-null      datetime64[ns]
 2   uf          223 non-null    object        
 3   canal       223 non-null    object        
 4   categoria   223 non-null    object        
 5   prod

/tmp/ipykernel_986/3924075162.py:1: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df['data'] = pd.to_datetime(df['data'], infer_datetime_format=True, errors='coerce')


### Reflexão curta
Explique:
1. Por que deixar `data` como texto pode quebrar análises temporais?
a ordenação será lexicográfica
2. Por que deixar `receita` como string pode distorcer cálculos?
operações matemáticas não podem ser realizadas em strings


## 5. Batalha 2 — O enigma dos valores ausentes

A aula reforça que valores ausentes não devem ser tratados automaticamente; a decisão precisa ser justificada. fileciteturn2file0

### Tarefa
1. Mapeie os valores ausentes por coluna
2. Identifique quais colunas críticas têm nulos
3. Defina uma estratégia para cada caso:
   - remover linhas?
   - preencher?
   - manter?
4. Justifique cada escolha

### Dica
Evite decisões automáticas sem análise de contexto.


In [7]:

print("\nContagem atualizada de valores nulos por coluna:")
print(df.isnull().sum())


critical_columns_with_nulls = ['data', 'uf', 'canal', 'categoria', 'receita', 'lucro']
rows_with_critical_nulls = df[df[critical_columns_with_nulls].isnull().any(axis=1)]

print(f"\nNúmero de linhas com valores nulos em colunas críticas ({critical_columns_with_nulls}): {len(rows_with_critical_nulls)}")
display(rows_with_critical_nulls.head())

print("\nRemovendo linhas com valores nulos em colunas críticas...")
df.dropna(subset=critical_columns_with_nulls, inplace=True)

print("Preenchendo valores nulos na coluna 'observacao' com 'Sem Observação'...")
df['observacao'].fillna('Sem Observação', inplace=True)

print("\nContagem de valores nulos após tratamento:")
print(df.isnull().sum())

print("\nInformações finais do DataFrame após tratamento de nulos:")
df.info()


Contagem atualizada de valores nulos por coluna:
pedido_id     0
data          0
uf            0
canal         0
categoria     0
produto       0
cliente_id    0
quantidade    0
receita       0
lucro         0
observacao    0
dtype: int64

Número de linhas com valores nulos em colunas críticas (['data', 'uf', 'canal', 'categoria', 'receita', 'lucro']): 0


,pedido_id,data,uf,canal,categoria,produto,cliente_id,quantidade,receita,lucro,observacao



Removendo linhas com valores nulos em colunas críticas...
Preenchendo valores nulos na coluna 'observacao' com 'Sem Observação'...

Contagem de valores nulos após tratamento:
pedido_id     0
data          0
uf            0
canal         0
categoria     0
produto       0
cliente_id    0
quantidade    0
receita       0
lucro         0
observacao    0
dtype: int64

Informações finais do DataFrame após tratamento de nulos:
<class 'pandas.core.frame.DataFrame'>
Index: 3 entries, 0 to 154
Data columns (total 11 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   pedido_id   3 non-null      int64         
 1   data        3 non-null      datetime64[ns]
 2   uf          3 non-null      object        
 3   canal       3 non-null      object        
 4   categoria   3 non-null      object        
 5   produto     3 non-null      object        
 6   cliente_id  3 non-null      object        
 7   quantidade  3 non-null      int64        

/tmp/ipykernel_986/3366978778.py:15: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['observacao'].fillna('Sem Observação', inplace=True)


## 6. Batalha 3 — O ataque dos clones (duplicidades)

A aula alerta que nem toda duplicidade é erro automático: pode haver compra legítima repetida ou dupla inserção do sistema. fileciteturn2file0

### Tarefa
1. Investigue se há linhas duplicadas
2. Analise se a duplicidade parece nociva para os KPIs
3. Escolha uma estratégia:
   - remover duplicidades totais?
   - remover apenas com base em certas colunas?
   - manter?
4. Justifique a decisão


In [8]:
print("\n--- Verificação de Duplicidades ---")

total_duplicates = df.duplicated().sum()
print(f"Número de linhas totalmente duplicadas (todas as colunas): {total_duplicates}")

if total_duplicates > 0:
    print("Linhas totalmente duplicadas encontradas:")
    display(df[df.duplicated(keep=False)])

duplicidade_pedido_id = df.duplicated(subset=['pedido_id']).sum()
print(f"Número de 'pedido_id' duplicados: {duplicidade_pedido_id}")

if duplicidade_pedido_id > 0:
    print("Linhas com 'pedido_id' duplicados encontradas:")
    display(df[df.duplicated(subset=['pedido_id'], keep=False)])

if total_duplicates == 0 and duplicidade_pedido_id == 0:
    print("\nNão foram encontradas duplicidades significativas no DataFrame atual. Nenhuma ação de remoção é necessária.")
    print("Justificativa: Com o DataFrame atual contendo apenas 3 linhas, e sem duplicidades exatas ou de 'pedido_id', não há necessidade de remoção.")
else:
    print("\nDuplicidades encontradas. Removendo duplicidades baseadas em 'pedido_id'...")
    df.drop_duplicates(subset=['pedido_id'], inplace=True)
    print("Duplicidades removidas. Nova contagem de linhas:", len(df))
    print("Justificativa: 'pedido_id' deve ser um identificador único para cada transação. Duplicatas nesta coluna indicam erro de registro e, portanto, devem ser removidas para garantir a integridade dos dados.")

print("\nInformações finais do DataFrame após tratamento de duplicidades:")
df.info()


--- Verificação de Duplicidades ---
Número de linhas totalmente duplicadas (todas as colunas): 0
Número de 'pedido_id' duplicados: 0

Não foram encontradas duplicidades significativas no DataFrame atual. Nenhuma ação de remoção é necessária.
Justificativa: Com o DataFrame atual contendo apenas 3 linhas, e sem duplicidades exatas ou de 'pedido_id', não há necessidade de remoção.

Informações finais do DataFrame após tratamento de duplicidades:
<class 'pandas.core.frame.DataFrame'>
Index: 3 entries, 0 to 154
Data columns (total 11 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   pedido_id   3 non-null      int64         
 1   data        3 non-null      datetime64[ns]
 2   uf          3 non-null      object        
 3   canal       3 non-null      object        
 4   categoria   3 non-null      object        
 5   produto     3 non-null      object        
 6   cliente_id  3 non-null      object        
 7   quantidade  3 non

## 7. Padronização de categorias

Os slides mostram que categorias duplicadas ou mal escritas podem distorcer rankings e filtros. fileciteturn2file0

### Tarefa
Inspecione e padronize, se necessário:
- `uf`
- `canal`
- `categoria`

### Pense em:
- maiúsculas e minúsculas
- espaços extras
- acentuação / variações
- categorias equivalentes escritas de formas diferentes


In [10]:
print("Valores únicos da coluna 'uf' antes da padronização:")
print(df['uf'].unique())

print("\nValores únicos da coluna 'canal' antes da padronização:")
print(df['canal'].unique())

print("\nValores únicos da coluna 'categoria' antes da padronização:")
print(df['categoria'].unique())

df['uf'] = df['uf'].astype(str).str.lower().str.strip()

df['canal'] = df['canal'].astype(str).str.lower().str.strip()

df['categoria'] = df['categoria'].astype(str).str.lower().str.strip()

print("\nValores únicos da coluna 'uf' após padronização:")
print(df['uf'].unique())

print("\nValores únicos da coluna 'canal' após padronização:")
print(df['canal'].unique())

print("\nValores únicos da coluna 'categoria' após padronização:")
print(df['categoria'].unique())


Valores únicos da coluna 'uf' antes da padronização:
['sp' 'mg' 'rs']

Valores únicos da coluna 'canal' antes da padronização:
['loja física' 'marketplace']

Valores únicos da coluna 'categoria' antes da padronização:
['informática' 'telefonia']

Valores únicos da coluna 'uf' após padronização:
['sp' 'mg' 'rs']

Valores únicos da coluna 'canal' após padronização:
['loja física' 'marketplace']

Valores únicos da coluna 'categoria' após padronização:
['informática' 'telefonia']


## 8. Subindo de nível — Criação de variáveis derivadas

Depois da limpeza, é hora de enriquecer a base com variáveis úteis para análise. fileciteturn2file0

### Tarefa
Crie, se possível:
- `ano`
- `mes`
- `ano_mes`
- `margem_lucro`

### Atenção
A criação de `margem_lucro` exige cuidado com divisões por zero e possíveis valores infinitos.


In [11]:
df['ano'] = df['data'].dt.year

df['mes'] = df['data'].dt.month

df['ano_mes'] = df['data'].dt.strftime('%Y-%m')

df['margem_lucro'] = np.where(df['receita'] != 0, df['lucro'] / df['receita'], 0)

print("DataFrame com variáveis derivadas adicionadas:")
display(df.head())

print("\nVerificando os tipos de dados após a criação das novas colunas:")
df.info()


DataFrame com variáveis derivadas adicionadas:


,pedido_id,data,uf,canal,categoria,produto,cliente_id,quantidade,receita,lucro,observacao,ano,mes,ano_mes,margem_lucro
0,1000,2024-04-29,sp,loja física,informática,Notebook Pro,C134,1,4535.11,1289.83,ok,2024,4,2024-04,0.284410
101,1101,2024-10-21,mg,marketplace,telefonia,Smartphone X,C151,4,11466.36,2087.05,Sem Observação,2024,10,2024-10,0.182015
154,1154,2024-09-30,rs,loja física,informática,Notebook Pro,C155,1,3998.83,641.54,Sem Observação,2024,9,2024-09,0.160432



Verificando os tipos de dados após a criação das novas colunas:
<class 'pandas.core.frame.DataFrame'>
Index: 3 entries, 0 to 154
Data columns (total 15 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   pedido_id     3 non-null      int64         
 1   data          3 non-null      datetime64[ns]
 2   uf            3 non-null      object        
 3   canal         3 non-null      object        
 4   categoria     3 non-null      object        
 5   produto       3 non-null      object        
 6   cliente_id    3 non-null      object        
 7   quantidade    3 non-null      int64         
 8   receita       3 non-null      float64       
 9   lucro         3 non-null      float64       
 10  observacao    3 non-null      object        
 11  ano           3 non-null      int32         
 12  mes           3 non-null      int32         
 13  ano_mes       3 non-null      object        
 14  margem_lucro  3 non-null      fl

## 9. Seleção final — Menos é mais

A aula propõe que nem toda coluna precisa seguir para a base analítica final. fileciteturn2file0

### Tarefa
Crie um DataFrame final, por exemplo `df_clean` ou `df_dash`, contendo apenas as colunas estritamente necessárias para análises de negócio.

**Sugestão de raciocínio:**
- Quais colunas ajudam a responder perguntas de negócio?
- Quais colunas são operacionais, auxiliares ou dispensáveis?
- O que precisa existir para as próximas aulas de visualização e dashboard?


In [12]:
colunas_essenciais = [
    'pedido_id',
    'data',
    'uf',
    'canal',
    'categoria',
    'produto',
    'cliente_id',
    'quantidade',
    'receita',
    'lucro',
    'ano',
    'mes',
    'ano_mes',
    'margem_lucro'
]

df_clean = df[colunas_essenciais].copy()

print("DataFrame final (df_clean) criado com as colunas essenciais:")
display(df_clean.head())

print("\nInformações do DataFrame final (df_clean):")
df_clean.info()


DataFrame final (df_clean) criado com as colunas essenciais:


,pedido_id,data,uf,canal,categoria,produto,cliente_id,quantidade,receita,lucro,ano,mes,ano_mes,margem_lucro
0,1000,2024-04-29,sp,loja física,informática,Notebook Pro,C134,1,4535.11,1289.83,2024,4,2024-04,0.284410
101,1101,2024-10-21,mg,marketplace,telefonia,Smartphone X,C151,4,11466.36,2087.05,2024,10,2024-10,0.182015
154,1154,2024-09-30,rs,loja física,informática,Notebook Pro,C155,1,3998.83,641.54,2024,9,2024-09,0.160432



Informações do DataFrame final (df_clean):
<class 'pandas.core.frame.DataFrame'>
Index: 3 entries, 0 to 154
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   pedido_id     3 non-null      int64         
 1   data          3 non-null      datetime64[ns]
 2   uf            3 non-null      object        
 3   canal         3 non-null      object        
 4   categoria     3 non-null      object        
 5   produto       3 non-null      object        
 6   cliente_id    3 non-null      object        
 7   quantidade    3 non-null      int64         
 8   receita       3 non-null      float64       
 9   lucro         3 non-null      float64       
 10  ano           3 non-null      int32         
 11  mes           3 non-null      int32         
 12  ano_mes       3 non-null      object        
 13  margem_lucro  3 non-null      float64       
dtypes: datetime64[ns](1), float64(3), int32(2), int64(2),

## 10. Validação final da base limpa

Antes de exportar, faça uma checagem final.

### Verifique:
- os tipos estão corretos?
- ainda há nulos problemáticos?
- ainda há duplicidades nocivas?
- existe algum infinito em `margem_lucro`?
- a base está pronta para ser usada em análises e dashboards?


In [ ]:
print("--- Validação Final da Base Limpa ---")

print("\n1. Verificando Tipos de Dados:")
df_clean.info()

print("\n2. Verificando Valores Nulos Problemáticos:")
nulos_finais = df_clean.isnull().sum()
print(nulos_finais[nulos_finais > 0])
if nulos_finais.sum() == 0:
    print("Nenhum valor nulo encontrado em df_clean.")
else:
    print("Ainda existem valores nulos. Revise as colunas acima.")

print("\n3. Verificando Duplicidades Nocivas:")
total_duplicates_clean = df_clean.duplicated().sum()
print(f"Número de linhas totalmente duplicadas em df_clean: {total_duplicates_clean}")
if total_duplicates_clean == 0:
    print("Nenhuma linha totalmente duplicada encontrada.")
else:
    print("Linhas totalmente duplicadas encontradas. Revise.")

duplicidade_pedido_id_clean = df_clean.duplicated(subset=['pedido_id']).sum()
print(f"Número de 'pedido_id' duplicados em df_clean: {duplicidade_pedido_id_clean}")
if duplicidade_pedido_id_clean == 0:
    print("Nenhum 'pedido_id' duplicado encontrado.")
else:
    print("Existe 'pedido_id' duplicado. Isso pode indicar um problema grave e deve ser investigado.")

print("\n4. Verificando Valores Infinitos em 'margem_lucro':")
infs_margem_lucro = np.isinf(df_clean['margem_lucro']).sum()
print(f"Número de valores infinitos na coluna 'margem_lucro': {infs_margem_lucro}")
if infs_margem_lucro == 0:
    print("Nenhum valor infinito encontrado na coluna 'margem_lucro'.")
else:
    print("Valores infinitos encontrados em 'margem_lucro'. Revise a lógica de cálculo.")

print("\n5. Conclusão sobre a prontidão da base:")
if nulos_finais.sum() == 0 and total_duplicates_clean == 0 and duplicidade_pedido_id_clean == 0 and infs_margem_lucro == 0:
    print("A base `df_clean` parece estar limpa e pronta para uso em análises e dashboards.")
else:
    print("A base `df_clean` ainda apresenta inconsistências e precisa de revisão.")


In [14]:
df_clean.to_csv('vendas_brasil_aula3_clean.csv', index=False)
print("Base 'vendas_brasil_aula3_clean.csv' exportada com sucesso!")

Base 'vendas_brasil_aula3_clean.csv' exportada com sucesso!


## 12. Conclusão e registro reflexivo

1.  Quais foram os principais problemas de qualidade encontrados?
      Tipos de Dados Incorretos, valores Ausentes, duplicidades

2.  Qual decisão de limpeza foi mais difícil?
    foi o tratamento dos valores nulos

3.  Que impacto essas falhas poderiam causar em um dashboard?
    cálculos Incorretos, visualizações enganosas, análises imprecisas, problemas de performance

4.  Por que a etapa de limpeza é considerada a "fundação invisível" do projeto?
    É um trabalho fundamental que consome tempo e exige decisões cuidadosas, mas que raramente é percebido diretamente pelo usuário final do dashboard. No entanto, é ela que garante a integridade, precisão e confiança em todas as informações apresentadas.